# Getting Started with Dual-Branch Malware Detection

This notebook demonstrates how to use the dual-branch malware detection model.

## 1. Setup and Imports

In [ ]:
import sys
sys.path.append('..')

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

from models import CNNBranch, LSTMBranch, DualBranchModel
from utils import (
    MalwareDataset, create_data_loaders,
    preprocess_binary, preprocess_sequence,
    calculate_metrics, plot_confusion_matrix, plot_roc_curve
)

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. Model Architecture Overview

In [ ]:
# Initialize the dual-branch model
model = DualBranchModel(
    cnn_config={
        'input_channels': 1,
        'num_classes': 2,
        'feature_dim': 256
    },
    lstm_config={
        'input_size': 100,
        'hidden_size': 128,
        'num_layers': 2,
        'feature_dim': 256,
        'bidirectional': True
    },
    num_classes=2,
    fusion_method='concat'
)

model = model.to(device)
print(model)

## 3. Load and Prepare Data

**Note**: Replace this section with your actual Kaggle dataset loading code.

In [ ]:
# Example: Generate dummy data for demonstration
# Replace this with your actual data loading

num_samples = 1000
img_size = 256
seq_length = 1000
input_size = 100

# Dummy CNN data (binary visualizations)
cnn_data = np.random.randn(num_samples, 1, img_size, img_size).astype(np.float32)

# Dummy LSTM data (sequential features)
lstm_data = np.random.randn(num_samples, seq_length, input_size).astype(np.float32)

# Dummy labels (0: benign, 1: malware)
labels = np.random.randint(0, 2, num_samples)

# Split into train and validation
split_idx = int(0.8 * num_samples)

train_cnn = cnn_data[:split_idx]
train_lstm = lstm_data[:split_idx]
train_labels = labels[:split_idx]

val_cnn = cnn_data[split_idx:]
val_lstm = lstm_data[split_idx:]
val_labels = labels[split_idx:]

print(f'Training samples: {len(train_labels)}')
print(f'Validation samples: {len(val_labels)}')

## 4. Create Data Loaders

In [ ]:
batch_size = 32

train_loader, val_loader = create_data_loaders(
    train_cnn, train_lstm, train_labels,
    val_cnn, val_lstm, val_labels,
    batch_size=batch_size,
    num_workers=0,  # Set to 0 for Windows compatibility
    shuffle=True
)

print(f'Number of training batches: {len(train_loader)}')
print(f'Number of validation batches: {len(val_loader)}')

## 5. Training Setup

In [ ]:
# Loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)

# Training parameters
num_epochs = 10
best_val_loss = float('inf')

## 6. Training Loop

In [ ]:
from tqdm import tqdm

history = {
    'train_loss': [],
    'train_accuracy': [],
    'val_loss': [],
    'val_accuracy': []
}

for epoch in range(num_epochs):
    # Training phase
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0
    
    for batch in tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs} [Train]'):
        cnn_input = batch['cnn_input'].to(device)
        lstm_input = batch['lstm_input'].to(device)
        labels = batch['label'].to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(cnn_input, lstm_input)
        loss = criterion(outputs, labels)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Statistics
        train_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()
    
    # Validation phase
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f'Epoch {epoch+1}/{num_epochs} [Val]'):
            cnn_input = batch['cnn_input'].to(device)
            lstm_input = batch['lstm_input'].to(device)
            labels = batch['label'].to(device)
            
            outputs = model(cnn_input, lstm_input)
            loss = criterion(outputs, labels)
            
            val_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()
    
    # Calculate metrics
    train_loss /= len(train_loader)
    train_acc = 100 * train_correct / train_total
    val_loss /= len(val_loader)
    val_acc = 100 * val_correct / val_total
    
    # Update history
    history['train_loss'].append(train_loss)
    history['train_accuracy'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_accuracy'].append(val_acc)
    
    # Learning rate scheduling
    scheduler.step(val_loss)
    
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), '../checkpoints/best_model.pth')
    
    print(f'Epoch {epoch+1}/{num_epochs}:')
    print(f'  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%')
    print(f'  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%')
    print()

## 7. Visualize Training History

In [ ]:
from utils.metrics import plot_training_history

plot_training_history(history, metrics=['loss', 'accuracy'])

## 8. Evaluation

In [ ]:
# Load best model
model.load_state_dict(torch.load('../checkpoints/best_model.pth'))
model.eval()

# Get predictions
all_preds = []
all_labels = []
all_probs = []

with torch.no_grad():
    for batch in val_loader:
        cnn_input = batch['cnn_input'].to(device)
        lstm_input = batch['lstm_input'].to(device)
        labels = batch['label'].to(device)
        
        outputs = model(cnn_input, lstm_input)
        probs = torch.softmax(outputs, dim=1)
        _, predicted = torch.max(outputs.data, 1)
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_probs.extend(probs[:, 1].cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

## 9. Calculate Metrics

In [ ]:
from utils.metrics import print_classification_report

metrics = calculate_metrics(all_labels, all_preds, all_probs)

print('\nEvaluation Metrics:')
print(f"Accuracy: {metrics['accuracy']:.4f}")
print(f"Precision: {metrics['precision']:.4f}")
print(f"Recall: {metrics['recall']:.4f}")
print(f"F1-Score: {metrics['f1_score']:.4f}")
print(f"AUC: {metrics['auc']:.4f}")
print(f"Specificity: {metrics['specificity']:.4f}")
print(f"Sensitivity: {metrics['sensitivity']:.4f}")

print_classification_report(all_labels, all_preds)

## 10. Visualize Results

In [ ]:
# Confusion Matrix
plot_confusion_matrix(all_labels, all_preds, class_names=['Benign', 'Malware'])

# ROC Curve
plot_roc_curve(all_labels, all_probs)

## 11. Feature Visualization (Optional)

In [ ]:
# Extract features from a sample
sample_batch = next(iter(val_loader))
cnn_input = sample_batch['cnn_input'][:1].to(device)
lstm_input = sample_batch['lstm_input'][:1].to(device)

with torch.no_grad():
    cnn_features, lstm_features, fused_features = model.get_branch_features(cnn_input, lstm_input)

print(f'CNN Features shape: {cnn_features.shape}')
print(f'LSTM Features shape: {lstm_features.shape}')
print(f'Fused Features shape: {fused_features.shape}')

## Next Steps

1. Replace dummy data with your actual Kaggle dataset
2. Experiment with different hyperparameters
3. Try different fusion methods ('concat', 'add', 'attention')
4. Implement data augmentation
5. Add early stopping
6. Experiment with different CNN architectures
7. Tune LSTM parameters
8. Perform cross-validation